# Fase 2 - Comprensión de los Datos
## Sección 09: Análisis Geoespacial

**Notebook:** notebooks/09_EDA_geoespacial.ipynb
**Responsable:** Sofia | **Apoyo:** Steve
**Objetivo:** Validar coordenadas geográficas de propiedades y mapear su distribución en Colombia.


## Configuración inicial

In [ ]:
from config import *


---
## Sección 09: Análisis Geoespacial
## Validación de coordenadas

### Cargar datasets con lat/lon

In [ ]:
GEO_CFG = {
    'A1': ('A1_colombia_housing_properties.csv', 'lat', 'lon', 'l3'),
    'A5': ('A5_medellin_properties_2023.csv', 'latitude', 'longitude', None),
}

geo_data = []
for fid, (fname, latcol, loncol, ccol) in GEO_CFG.items():
    df = pd.read_csv(os.path.join(RAW, fname), encoding='utf-8-sig', low_memory=False)
    df['lat'] = pd.to_numeric(df[latcol], errors='coerce')
    df['lon'] = pd.to_numeric(df[loncol], errors='coerce')
    df['fuente'] = fid
    if ccol and ccol in df.columns:
        df['ciudad'] = df[ccol].astype(str).str.strip().str.title()
    else:
        df['ciudad'] = 'Medellin' if fid == 'A5' else None
    geo_data.append(df[['lat','lon','fuente','ciudad']])
    print(f'{fid}: {df.shape[0]:>8,} filas  lat={latcol}  lon={loncol}')

df_geo = pd.concat(geo_data, ignore_index=True)
print(f'\nTotal: {len(df_geo):,} registros')


### Validar rango de coordenadas para Colombia

In [ ]:
LAT_MIN, LAT_MAX = -4.5, 13.0
LON_MIN, LON_MAX = -79.0, -66.0

df_geo['lat_valida'] = df_geo['lat'].between(LAT_MIN, LAT_MAX)
df_geo['lon_valida'] = df_geo['lon'].between(LON_MIN, LON_MAX)
df_geo['coords_validas'] = df_geo['lat_valida'] & df_geo['lon_valida']

print('--- RANGO VALIDO PARA COLOMBIA ---')
print(f'  Latitud:  {LAT_MIN} a {LAT_MAX}')
print(f'  Longitud: {LON_MIN} a {LON_MAX}')

n_total = len(df_geo)
n_con_coords = df_geo['lat'].notna().sum()
n_validas = df_geo['coords_validas'].sum()

print(f'\nRegistros totales:          {n_total:>10,}')
print(f'Con coordenadas:            {n_con_coords:>10,} ({n_con_coords/n_total*100:5.1f}%)')
print(f'Coordenadas en rango valido: {n_validas:>10,} ({n_validas/n_total*100:5.1f}%)')

# Invalidas
invalidas = df_geo[df_geo['lat'].notna() & ~df_geo['coords_validas']]
print(f'\nCoordenadas fuera de rango: {len(invalidas)}')
if len(invalidas) > 0:
    print('Muestra de coordenadas invalidas:')
    display(invalidas[['lat','lon','fuente','ciudad']].dropna().head(10))


### % de cobertura geoespacial por ciudad (A1)

In [ ]:
city_geo = df_geo[df_geo['fuente'] == 'A1'].groupby('ciudad').agg(
    total=('lat', 'count'),
    con_coords=('lat', lambda x: x.notna().sum()),
    validas=('coords_validas', 'sum')
)
city_geo['%_con_coords'] = (city_geo['con_coords'] / city_geo['total'] * 100).round(1)
city_geo['%_validas'] = (city_geo['validas'] / city_geo['total'] * 100).round(1)
city_geo = city_geo[city_geo['total'] >= 100].sort_values('total', ascending=False)
print('--- COBERTURA GEOESPACIAL POR CIUDAD (>=100 registros) ---')
display(city_geo[['total','%_con_coords','%_validas']].head(15))


### Mapa de distribucion de puntos

In [ ]:
print('--- MAPA DE DISTRIBUCION ---')
print('Generando scatterplot geografico (lat vs lon)...')

valid = df_geo[df_geo['coords_validas']]
sample = valid.sample(min(20000, len(valid)), random_state=42)

plt.figure(figsize=(14, 10))
colors = {'A1': 'steelblue', 'A5': 'crimson'}
for fid in ['A1', 'A5']:
    sub = sample[sample['fuente'] == fid]
    plt.scatter(sub['lon'], sub['lat'], c=colors[fid], label=fid, alpha=0.3, s=2)

plt.xlabel('Longitud')
plt.ylabel('Latitud')
plt.title(f'Distribucion geoespacial de propiedades (muestra {len(sample):,} pts)')
plt.legend()
plt.grid(alpha=0.2)

# Limites Colombia
plt.xlim(LON_MIN, LON_MAX)
plt.ylim(LAT_MIN, LAT_MAX)

plt.tight_layout()
plt.savefig(os.path.join(FIGS, 'mapa_distribucion.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Mapa guardado en docs/figures/mapa_distribucion.png')

print("Grafico guardado en: mapa_distribucion.png")

fig.suptitle("Distribucion geoespacial de propiedades en Colombia (muestra) — solo 2 de 8 datasets tienen coordenadas (A1 y A5)", fontsize=14, y=1.02)


**Conclusion distribucion geoespacial:**
- Solo 2 de 8 datasets (A1 Properati, A5 Medellin) tienen coordenadas geograficas.
- La cobertura de A1 es nacional pero concentrada en Bogota-Medellin-Cali.
- A5 solo cubre Medellin (2023) — cobertura geografica limitada.


### Mapa de densidad (hexbin)

In [ ]:
plt.figure(figsize=(14, 10))
valid_a1 = df_geo[(df_geo['fuente']=='A1') & df_geo['coords_validas']]
sample_a1 = valid_a1.sample(min(50000, len(valid_a1)), random_state=42)
plt.hexbin(sample_a1['lon'], sample_a1['lat'], gridsize=60, cmap='Blues', alpha=0.8)
plt.colorbar(label='Conteo de propiedades')
plt.xlabel('Longitud')
plt.ylabel('Latitud')
plt.title('Densidad de propiedades - A1 Properati')
plt.xlim(LON_MIN, LON_MAX)
plt.ylim(LAT_MIN, LAT_MAX)
plt.tight_layout()
plt.savefig(os.path.join(FIGS, 'mapa_densidad.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Mapa de densidad guardado.')

print("Grafico guardado en: mapa_densidad.png")

fig.suptitle("Densidad de propiedades (hexbin) — A1 Properati, concentracion en ejes Bogota-Medellin-Cali", fontsize=14, y=1.02)


**Conclusion densidad geoespacial:**
- La densidad de propiedades se concentra en los ejes urbanos principales.
- Ciudades intermedias tienen poca representacion en terminos geoespaciales.
- Para Fase 3: usar coordenadas solo a nivel ciudad/region, no a nivel de barrio.


### Limitaciones geoespaciales

In [ ]:
print('--- LIMITACIONES GEOESPACIALES ---\n')
print('1. Solo 2 de 8 datasets del Grupo A tienen coordenadas (A1 y A5).')
print('2. A1 tiene cobertura nacional pero concentrada en ejes Bogota-Medellin-Cali.')
print('3. A5 solo cubre Medellin (2023).')
print(f'4. {invalidas.shape[0]} registros tienen coordenadas fuera del rango de Colombia.')
print('5. No hay datos de coordenadas para ciudades intermedias (Ibague, Cucuta, etc.).')
print('6. La calidad de lat/lon en A1 puede tener errores de geocodificacion.\n')
print('Recomendaciones:')
print('  - Usar coordenadas solo para analisis a nivel de ciudad/region.')
print('  - No usar para analisis a nivel de barrio por precision limitada.')
print('  - Considerar geocodificar direcciones en A2/A7 como mejora futura.')


---
## Resumen: Análisis Geoespacial

- Se validaron coordenadas geográficas de los datasets del Grupo A con lat/lon.
- Solo 2 de 8 datasets (A1 Properati, A5 Medellín) tienen coordenadas.
- Se generó mapa de distribución de puntos y mapa de densidad (hexbin).
- Se documentaron limitaciones geoespaciales y recomendaciones.

**Outputs generados:**
- `docs/figures/mapa_distribucion.png`
- `docs/figures/mapa_densidad.png`
